In [1]:
print("==========================================")
print("   TASK SCHEDULER & JOB RUNNER")
print("==========================================")
print("Python Full Stack Internship Project")
print("Features: Cron Jobs, Task Execution, SQLite, Monitoring")

   TASK SCHEDULER & JOB RUNNER
Python Full Stack Internship Project
Features: Cron Jobs, Task Execution, SQLite, Monitoring


In [2]:
!pip install flask apscheduler flask-cors -q

print("✅ Required packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 4.4 MB/s eta 0:00:00
✅ Required packages installed successfully!


In [3]:
from flask import Flask, jsonify, request, render_template_string
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger
from datetime import datetime
import sqlite3

app = Flask(__name__)

scheduler = BackgroundScheduler()
scheduler.start()

execution_history = []

print("✅ Flask application created")
print("✅ Scheduler started")

✅ Flask application created
✅ Scheduler started


In [5]:
def init_database():
    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            task_name TEXT NOT NULL,
            hour INTEGER NOT NULL,
            minute INTEGER NOT NULL,
            status TEXT DEFAULT 'Scheduled'
        )
    """)

    conn.commit()
    conn.close()

init_database()

print("✅ SQLite database initialized")

✅ SQLite database initialized


In [6]:
def run_task(task_name):
    start_time = datetime.now()

    print(f"▶️ Running task: {task_name}")
    print(f"⏰ Started: {start_time}")

    status = "Success"

    end_time = datetime.now()

    execution_history.append({
        "task": task_name,
        "status": status,
        "started": str(start_time),
        "finished": str(end_time)
    })

    print(f"✅ Task completed: {task_name}")

    return status


def create_cron_task(task_name, hour, minute):
    job_id = "cron_" + task_name.replace(" ", "_")

    scheduler.add_job(
        run_task,
        CronTrigger(hour=hour, minute=minute),
        args=[task_name],
        id=job_id,
        replace_existing=True
    )

    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute(
        "INSERT INTO tasks (task_name, hour, minute, status) VALUES (?, ?, ?, ?)",
        (task_name, hour, minute, "Scheduled")
    )

    conn.commit()
    conn.close()

    return {
        "task": task_name,
        "hour": hour,
        "minute": minute,
        "status": "Scheduled"
    }


print("✅ Task execution function ready")
print("✅ Cron task creation function ready")

✅ Task execution function ready
✅ Cron task creation function ready


In [7]:
HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Task Scheduler & Job Runner</title>

    <style>
        body {
            font-family: Arial;
            margin: 40px;
            background: #f5f5f5;
        }

        .container {
            max-width: 800px;
            margin: auto;
            background: white;
            padding: 30px;
            border-radius: 12px;
        }

        input, button {
            padding: 10px;
            margin: 5px;
        }

        button {
            cursor: pointer;
        }

        .box {
            padding: 15px;
            margin-top: 20px;
            border: 1px solid #ddd;
            border-radius: 8px;
        }
    </style>
</head>

<body>

<div class="container">

    <h1>⏰ Task Scheduler & Job Runner</h1>

    <div class="box">
        <h2>➕ Create Cron Job</h2>

        <form action="/create" method="post">

            <input
                type="text"
                name="task_name"
                placeholder="Task name"
                required
            >

            <input
                type="number"
                name="hour"
                min="0"
                max="23"
                placeholder="Hour"
                required
            >

            <input
                type="number"
                name="minute"
                min="0"
                max="59"
                placeholder="Minute"
                required
            >

            <button type="submit">
                Create Cron Job
            </button>

        </form>
    </div>

    <div class="box">
        <h2>📋 Scheduled Tasks</h2>
        <a href="/tasks">View Tasks</a>
    </div>

    <div class="box">
        <h2>📊 Execution History</h2>
        <a href="/history">View History</a>
    </div>

</div>

</body>
</html>
"""

print("✅ Dashboard HTML ready")

✅ Dashboard HTML ready


In [8]:
@app.route("/")
def home():
    return """
    <h1>⏰ Task Scheduler & Job Runner</h1>
    <p>Application is running successfully.</p>
    <a href="/dashboard">Open Dashboard</a>
    """


@app.route("/dashboard")
def dashboard():
    return render_template_string(HTML)


@app.route("/create", methods=["POST"])
def create():
    task_name = request.form["task_name"]
    hour = int(request.form["hour"])
    minute = int(request.form["minute"])

    result = create_cron_task(task_name, hour, minute)

    return jsonify(result)


@app.route("/tasks")
def tasks_api():
    conn = sqlite3.connect("tasks.db")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT id, task_name, hour, minute, status FROM tasks"
    )

    rows = cursor.fetchall()
    conn.close()

    tasks = []

    for row in rows:
        tasks.append({
            "id": row[0],
            "task": row[1],
            "hour": row[2],
            "minute": row[3],
            "status": row[4]
        })

    return jsonify(tasks)


@app.route("/history")
def history_api():
    return jsonify(execution_history)


print("✅ All Flask routes registered")
print("✅ /dashboard ready")
print("✅ /tasks ready")
print("✅ /history ready")

✅ All Flask routes registered
✅ /dashboard ready
✅ /tasks ready
✅ /history ready


In [9]:
print("🧪 Testing task execution...")
print("-" * 40)

run_task("Daily Backup")

print("-" * 40)
print("📊 Execution History")

for record in execution_history:
    print("Task:", record["task"])
    print("Status:", record["status"])
    print("Started:", record["started"])
    print("Finished:", record["finished"])
    print("-" * 40)

🧪 Testing task execution...
----------------------------------------
▶️ Running task: Daily Backup
⏰ Started: 2026-10-07 04:54:54.546367
✅ Task completed: Daily Backup
----------------------------------------
📊 Execution History
Task: Daily Backup
Status: Success
Started: 2026-10-07 04:54:54.546367
Finished: 2026-10-07 04:54:54.546429
----------------------------------------


In [10]:
from datetime import datetime, timedelta

future_time = datetime.now() + timedelta(minutes=2)

hour = future_time.hour
minute = future_time.minute

result = create_cron_task(
    "Automatic Test Job",
    hour,
    minute
)

print("✅ Cron job created")
print(result)
print(f"⏰ Scheduled for {hour:02d}:{minute:02d}")

✅ Cron job created
{'task': 'Automatic Test Job', 'hour': 4, 'minute': 57, 'status': 'Scheduled'}
⏰ Scheduled for 04:57


In [12]:
import threading

def start_server():
    app.run(
        host="0.0.0.0",
        port=5001,
        debug=False,
        use_reloader=False
    )

server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

print("🚀 Flask server started successfully!")
print("📱 Dashboard is running on port 5001")

🚀 Flask server started successfully!
📱 Dashboard is running on port 5001
 * Serving Flask app '__main__'
